In [2]:
!pip -q install requests beautifulsoup4 pandas rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.3 MB/s eta 0:00:00


In [5]:
from __future__ import annotations

import re
import json
import time
import html as html_lib
from datetime import datetime, timedelta, date
from typing import Dict, List, Optional, Tuple
from urllib.parse import urljoin, quote

import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag
from rapidfuzz import fuzz
from IPython.display import display, HTML


# =========================
# CONFIG
# =========================
AMC_BASE = "https://www.amctheatres.com"
RT_BASE  = "https://www.rottentomatoes.com"

THEATRES = [
    {"name": "AMC Tustin 14 @ The District",
     "url": "https://www.amctheatres.com/movie-theatres/los-angeles/amc-tustin-14-at-the-district/showtimes"},
    {"name": "AMC Orange 30",
     "url": "https://www.amctheatres.com/movie-theatres/los-angeles/amc-orange-30/showtimes"},
    {"name": "AMC Bay Street 16",
     "url": "https://www.amctheatres.com/movie-theatres/showtimes/amc-bay-street-16/showtimes"},
    {"name": "AMC Woodbridge 5",
     "url": "https://www.amctheatres.com/movie-theatres/los-angeles/amc-woodbridge-5/showtimes"},
]

# Optional: pin a specific weekend Saturday (YYYY-MM-DD) for testing
OVERRIDE_SATURDAY = None  # e.g. "2025-12-20"

# Turn on to see RT URL attempts
DEBUG_RT = False


# =========================
# REGEX / CONSTANTS
# =========================
SHOWTIME_HREF_RE = re.compile(r"^/showtimes/\d+", re.I)
MOVIE_HREF_RE    = re.compile(r"^/movies/", re.I)
TIME_RE          = re.compile(r"(\d{1,2}:\d{2}\s*[ap]m)", re.I)

# Text fallbacks (handle normal % and fullwidth ％)
RT_TOMA_TXT_1 = re.compile(r"(\d{1,3})\s*[％%]\s*Tomatometer", re.I)
RT_TOMA_TXT_2 = re.compile(r"Tomatometer\s*(\d{1,3})\s*[％%]", re.I)

RT_AUD_TXT_1  = re.compile(r"(\d{1,3})\s*[％%]\s*(Popcornmeter|Audience\s*Score)", re.I)
RT_AUD_TXT_2  = re.compile(r"(Popcornmeter|Audience\s*Score)\s*(\d{1,3})\s*[％%]", re.I)

# JSON-ish fallbacks
RT_TOMA_JSON_1 = re.compile(r'"tomatometerScore"\s*:\s*(\d{1,3})', re.I)
RT_TOMA_JSON_2 = re.compile(r'"tomatometerScore"\s*:\s*\{[^}]{0,300}?"value"\s*:\s*(\d{1,3})', re.I)

RT_AUD_JSON_1  = re.compile(r'"audienceScore"\s*:\s*(\d{1,3})', re.I)
RT_AUD_JSON_2  = re.compile(r'"audienceScore"\s*:\s*\{[^}]{0,300}?"value"\s*:\s*(\d{1,3})', re.I)
RT_AUD_JSON_3  = re.compile(r'"popcornmeter"\s*:\s*(\d{1,3})', re.I)
RT_AUD_JSON_4  = re.compile(r'"popcornmeter"\s*:\s*\{[^}]{0,300}?"score"\s*:\s*(\d{1,3})', re.I)


# =========================
# TIME / SESSION HELPERS
# =========================
def upcoming_weekend_pacific() -> Tuple[date, date]:
    """Return upcoming Saturday/Sunday in America/Los_Angeles."""
    try:
        from zoneinfo import ZoneInfo
        today = datetime.now(ZoneInfo("America/Los_Angeles")).date()
    except Exception:
        today = date.today()

    if OVERRIDE_SATURDAY:
        sat = datetime.strptime(OVERRIDE_SATURDAY, "%Y-%m-%d").date()
        return sat, sat + timedelta(days=1)

    wd = today.weekday()  # Mon=0 .. Sun=6
    sat = today + timedelta(days=(5 - wd)) if wd <= 5 else today + timedelta(days=6)
    return sat, sat + timedelta(days=1)

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": "https://www.google.com/",
    })
    return s

def fetch_html(session: requests.Session, url: str, params: dict | None = None, tries: int = 3) -> Tuple[int, str]:
    """Return (status_code, text). Retries a few times."""
    last_status, last_text = 0, ""
    for i in range(tries):
        try:
            r = session.get(url, params=params, timeout=30, allow_redirects=True)
            last_status = r.status_code
            last_text = r.text or ""
            if last_status == 200 and "Access Denied" not in last_text:
                return last_status, last_text
        except Exception:
            pass
        time.sleep(1.0 * (i + 1))
    return last_status, last_text


# =========================
# AMC SCRAPING
# =========================
def extract_time(txt: str) -> Optional[str]:
    m = TIME_RE.search(txt or "")
    if not m:
        return None
    return re.sub(r"\s+", " ", m.group(1).strip().lower())

def parse_time_for_sort(t: str) -> int:
    try:
        dt = datetime.strptime(t.strip().upper(), "%I:%M %p")
        return dt.hour * 60 + dt.minute
    except Exception:
        return 10**9

def is_a_list_excluded_near_tag(tag: Tag, max_parent_levels: int = 6) -> bool:
    """
    More reliable AMC parsing: only look within a limited number of ancestor levels
    so we don't accidentally match page-wide legends/footers.
    """
    lvl = 0
    for p in tag.parents:
        if not isinstance(p, Tag):
            continue
        txt = p.get_text(" ", strip=True).lower()
        if "excluded from a-list" in txt:
            return True
        lvl += 1
        if lvl >= max_parent_levels:
            break
    return False

def looks_like_format_label(txt: str) -> bool:
    t = (txt or "").strip()
    if not t or len(t) > 140:
        return False
    needles = ["imax", "dolby", "reald", "laser", "fan faves", "no trailers", "spoken", "subtitles", "open caption"]
    tl = t.lower()
    return any(n in tl for n in needles)

def scrape_amc_showtimes_for_date(session: requests.Session, theatre_name: str, showtimes_url: str, d: date) -> List[dict]:
    status, html_txt = fetch_html(session, showtimes_url, params={"date": d.isoformat()}, tries=3)
    if status != 200 or not html_txt:
        return []

    soup = BeautifulSoup(html_txt, "html.parser")

    # Find movie headings and walk until next heading
    movie_heads: List[Tuple[Tag, str]] = []
    for h in soup.find_all(["h1", "h2", "h3"]):
        a = h.find("a", href=MOVIE_HREF_RE)
        if a and a.get_text(strip=True):
            movie_heads.append((h, a.get_text(" ", strip=True)))

    if not movie_heads:
        return []

    out: List[dict] = []
    for idx, (h, title) in enumerate(movie_heads):
        stop_tag = movie_heads[idx + 1][0] if idx + 1 < len(movie_heads) else None
        current_format: Optional[str] = None

        el = h.next_element
        while el is not None and el is not stop_tag:
            if isinstance(el, Tag):
                if el.name in ["h4", "h5", "strong", "p", "div", "span"]:
                    txt = el.get_text(" ", strip=True)
                    if looks_like_format_label(txt):
                        current_format = txt

                if el.name == "a":
                    href = el.get("href") or ""
                    if SHOWTIME_HREF_RE.match(href):
                        t = extract_time(el.get_text(" ", strip=True))
                        if t:
                            a_list_excl = is_a_list_excluded_near_tag(el)  # <-- fixed parsing
                            out.append({
                                "movie_title": title,
                                "theatre": theatre_name,
                                "show_date": d.isoformat(),
                                "show_time": t,
                                "format_label": current_format,
                                "a_list_excluded": bool(a_list_excl),
                                "ticket_url": urljoin(AMC_BASE, href),
                            })
            el = el.next_element

    return out


# =========================
# TITLE NORMALIZATION
# =========================
def candidate_title_variants(title: str) -> List[str]:
    t = re.sub(r"\s+", " ", (title or "").strip())
    if not t:
        return []
    variants = [t]
    for pat in [
        r"\s+\d{1,3}(st|nd|rd|th)\s+anniversary\s*$",
        r"\s+\d{1,3}th\s+anniversary\s*$",
        r"\s+early\s+access(\s+event)?\s*$",
        r"\s+sneak\s+peek\s*$",
        r"\s+fan\s+event\s*$",
        r"\s+re-?release\s*$",
    ]:
        c = re.sub(pat, "", t, flags=re.I).strip()
        if c and c.lower() != t.lower():
            variants.append(c)

    seen, out = set(), []
    for v in variants:
        k = v.lower()
        if k not in seen:
            seen.add(k)
            out.append(v)
    return out


# =========================
# ROTTEN TOMATOES (SLUG-FIRST, ROBUST PARSE)
# =========================
def rt_slugify(title: str) -> str:
    s = (title or "").lower().strip()
    s = re.sub(r"['’]", "", s)          # drop apostrophes
    s = re.sub(r"[^a-z0-9]+", "_", s)   # non-alnum -> underscore
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def _int0_100(x: Optional[str]) -> Optional[int]:
    if not x:
        return None
    x = str(x).strip()
    if not re.fullmatch(r"\d{1,3}", x):
        return None
    n = int(x)
    if 0 <= n <= 100:
        return n
    return None

def rt_parse_scores(decoded_html: str, soup: BeautifulSoup) -> Tuple[Optional[int], Optional[int]]:
    """
    Try (1) component attributes (most reliable), (2) embedded JSON, (3) visible text.
    Returns (audience, critic).
    """
    audience = None
    critic = None

    # (1) Component attributes (RT often renders <score-board ... tomatometerscore=".." audiencescore="..">)
    attr_candidates = []
    for attr in ["tomatometerscore", "tomatometerScore", "tomatometerscoreallcritics", "tomatometerscoreall"]:
        tag = soup.find(attrs={attr: True})
        if tag:
            attr_candidates.append((attr, tag.get(attr)))

    for attr, val in attr_candidates:
        n = _int0_100(val)
        if n is not None:
            critic = n
            break

    aud_attr_candidates = []
    for attr in ["audiencescore", "audienceScore", "popcornmeterscore", "popcornmeterScore"]:
        tag = soup.find(attrs={attr: True})
        if tag:
            aud_attr_candidates.append((attr, tag.get(attr)))

    for attr, val in aud_attr_candidates:
        n = _int0_100(val)
        if n is not None:
            audience = n
            break

    if critic is not None and audience is not None:
        return audience, critic

    # (2) Embedded JSON patterns in the HTML
    if critic is None:
        for rx in [RT_TOMA_JSON_2, RT_TOMA_JSON_1]:
            m = rx.search(decoded_html)
            if m:
                critic = _int0_100(m.group(1))
                if critic is not None:
                    break

    if audience is None:
        for rx in [RT_AUD_JSON_4, RT_AUD_JSON_3, RT_AUD_JSON_2, RT_AUD_JSON_1]:
            m = rx.search(decoded_html)
            if m:
                audience = _int0_100(m.group(1))
                if audience is not None:
                    break

    if critic is not None and audience is not None:
        return audience, critic

    # (3) Visible text fallback
    full_text = soup.get_text(" ", strip=True)
    full_text = re.sub(r"\s+", " ", full_text)

    # try to anchor around H1
    h1 = soup.find("h1")
    tail = full_text
    if h1:
        anchor = h1.get_text(" ", strip=True)
        if anchor:
            pos = full_text.lower().find(anchor.lower())
            if pos != -1:
                tail = full_text[pos:pos + 50000]

    if critic is None:
        m = (RT_TOMA_TXT_1.search(tail) or RT_TOMA_TXT_2.search(tail) or
             RT_TOMA_TXT_1.search(full_text) or RT_TOMA_TXT_2.search(full_text))
        if m:
            critic = _int0_100(m.group(1))

    if audience is None:
        m = RT_AUD_TXT_1.search(tail)
        if m:
            audience = _int0_100(m.group(1))
        else:
            m = RT_AUD_TXT_2.search(tail)
            if m:
                audience = _int0_100(m.group(2))
            else:
                m = RT_AUD_TXT_1.search(full_text)
                if m:
                    audience = _int0_100(m.group(1))
                else:
                    m = RT_AUD_TXT_2.search(full_text)
                    if m:
                        audience = _int0_100(m.group(2))

    return audience, critic

def rt_get_scores(
    session: requests.Session,
    title: str,
    cache: Dict[str, Tuple[Optional[int], Optional[int], Optional[str]]],
    debug: bool = False
) -> Tuple[Optional[int], Optional[int], Optional[str]]:
    """
    SLUG-FIRST:
      title -> /m/<slug>
    Returns (rt_audience, rt_critic, rt_url).
    """
    key = (title or "").lower().strip()
    if key in cache:
        return cache[key]

    for q in candidate_title_variants(title):
        slug = rt_slugify(q)
        if not slug:
            continue

        rt_url = f"{RT_BASE}/m/{slug}"
        status, raw_html = fetch_html(session, rt_url, params=None, tries=3)

        if debug:
            print(f"[RT] {q} -> {rt_url} status={status} len={len(raw_html or '')}")

        if status != 200 or not raw_html:
            continue

        if "page not found" in raw_html.lower():
            continue

        decoded = html_lib.unescape(raw_html)
        soup = BeautifulSoup(decoded, "html.parser")

        aud, crit = rt_parse_scores(decoded, soup)
        cache[key] = (aud, crit, rt_url)
        return cache[key]

    cache[key] = (None, None, None)
    return cache[key]


# =========================
# IMDB (rating + runtime)
# =========================
def imdb_suggest_id(session: requests.Session, title: str) -> Optional[str]:
    q = (title or "").strip()
    if not q:
        return None
    first = q[0].lower()
    url = f"https://v2.sg.media-imdb.com/suggestion/{first}/{quote(q)}.json"
    try:
        r = session.get(url, timeout=20)
        r.raise_for_status()
        data = r.json()
    except Exception:
        return None

    items = data.get("d") or []
    best_id, best_score = None, -1

    for it in items:
        imdb_id = it.get("id")
        label = (it.get("l") or "").strip()
        kind  = (it.get("q") or "").lower()
        if not imdb_id or not imdb_id.startswith("tt") or not label:
            continue
        score = fuzz.token_set_ratio(label.lower(), q.lower())
        if kind in {"feature", "movie"}:
            score += 10
        if score > best_score:
            best_score = score
            best_id = imdb_id

    return best_id

_ISO8601_DUR_RE = re.compile(r"^PT(?:(\d+)H)?(?:(\d+)M)?$", re.I)

def parse_iso8601_duration_to_minutes(dur: Optional[str]) -> Optional[int]:
    if not dur:
        return None
    s = str(dur).strip().upper()
    m = _ISO8601_DUR_RE.match(s)
    if not m:
        return None
    h = int(m.group(1) or 0)
    mins = int(m.group(2) or 0)
    total = h * 60 + mins
    return total if total > 0 else None

def fmt_runtime(mins: Optional[int]) -> Optional[str]:
    if mins is None:
        return None
    h, m = divmod(int(mins), 60)
    if h and m:
        return f"{h}h {m}m"
    if h:
        return f"{h}h"
    return f"{m}m"

def imdb_rating_and_runtime(
    session: requests.Session,
    title: str,
    cache: Dict[str, Tuple[Optional[float], Optional[int]]]
) -> Tuple[Optional[float], Optional[int]]:
    """
    Returns (imdb_rating, runtime_minutes).
    Runtime is parsed from JSON-LD "duration" (e.g. PT2H12M), with a small HTML fallback.
    """
    key0 = (title or "").lower().strip()
    if key0 in cache:
        return cache[key0]

    for q in candidate_title_variants(title):
        key = q.lower()
        if key in cache:
            cache[key0] = cache[key]
            return cache[key]

        imdb_id = imdb_suggest_id(session, q)
        if not imdb_id:
            cache[key] = (None, None)
            continue

        url = f"https://www.imdb.com/title/{imdb_id}/"
        try:
            r = session.get(url, timeout=20)
            r.raise_for_status()
            soup = BeautifulSoup(r.text, "html.parser")

            rating_val: Optional[float] = None
            runtime_min: Optional[int] = None

            # Primary: JSON-LD
            for s in soup.find_all("script", attrs={"type": "application/ld+json"}):
                try:
                    payload = json.loads(s.get_text(strip=True) or "{}")
                except Exception:
                    continue
                objs = payload if isinstance(payload, list) else [payload]
                for obj in objs:
                    if not isinstance(obj, dict):
                        continue

                    if rating_val is None:
                        agg = obj.get("aggregateRating") or {}
                        rv = agg.get("ratingValue")
                        if rv is not None:
                            try:
                                rating_val = float(rv)
                            except Exception:
                                pass

                    if runtime_min is None:
                        dur = obj.get("duration")
                        runtime_min = parse_iso8601_duration_to_minutes(dur) or runtime_min

                if rating_val is not None and runtime_min is not None:
                    break

            # Fallback: any element with datetime="PT.."
            if runtime_min is None:
                for t in soup.find_all(attrs={"datetime": re.compile(r"^PT", re.I)}):
                    runtime_min = parse_iso8601_duration_to_minutes(t.get("datetime"))
                    if runtime_min is not None:
                        break

            cache[key] = (rating_val, runtime_min)
            cache[key0] = cache[key]
            return cache[key]
        except Exception:
            cache[key] = (None, None)

    cache[key0] = (None, None)
    return cache[key0]


# =========================
# SORTING RULE (YOUR SPEC)
#   0) RT audience
#   1) IMDb (if no RT audience)
#   2) RT critic (if neither)
# =========================
def pick_primary(rt_aud: Optional[int], imdb: Optional[float], rt_crit: Optional[int]) -> Tuple[int, Optional[float], str]:
    """
    Returns (rank, score, source)
    rank: 0 audience, 1 imdb, 2 critic, 3 none
    score: audience / imdb*10 / critic
    """
    if rt_aud is not None:
        return 0, float(rt_aud), "RT_AUDIENCE"
    if imdb is not None:
        return 1, float(imdb) * 10.0, "IMDB"
    if rt_crit is not None:
        return 2, float(rt_crit), "RT_CRITIC"
    return 3, None, "NONE"


# =========================
# OUTPUT FORMATTING
# =========================
def short_fmt(s: Optional[str]) -> Optional[str]:
    if not s:
        return None
    s = re.sub(r"\s+", " ", s).strip()
    return s if len(s) <= 40 else s[:37] + "..."

def build_showtimes_cell(df_st: pd.DataFrame) -> str:
    lines = []
    for theatre in sorted(df_st["theatre"].unique()):
        df_th = df_st[df_st["theatre"] == theatre].copy()

        lines.append(f"{theatre}")
        for show_date in sorted(df_th["show_date"].unique()):
            df_d = df_th[df_th["show_date"] == show_date].copy()
            df_d["t_sort"] = df_d["show_time"].apply(parse_time_for_sort)
            df_d = df_d.sort_values("t_sort")

            times = []
            for _, r in df_d.iterrows():
                excl = "⛔" if bool(r["a_list_excluded"]) else ""
                fmt = short_fmt(r.get("format_label"))
                fmt_txt = f" [{fmt}]" if fmt else ""
                times.append(f"{r['show_time']}{excl}{fmt_txt}")

            lines.append(f"• {show_date}: " + ", ".join(times))

    return "\n".join(lines)

# =========================
# RUN
# =========================
sat, sun = upcoming_weekend_pacific()
dates = [sat, sun]

session = make_session()

# Scrape showtimes
showtimes: List[dict] = []
for th in THEATRES:
    for d in dates:
        try:
            showtimes.extend(scrape_amc_showtimes_for_date(session, th["name"], th["url"], d))
            time.sleep(0.2)
        except Exception as e:
            print(f"[WARN] Failed scraping {th['name']} {d}: {e}")

if not showtimes:
    raise RuntimeError("No showtimes parsed. AMC may be blocking requests or the page structure changed.")

df_show = pd.DataFrame(showtimes)

# Lookup scores (1 row per movie)
imdb_cache: Dict[str, Tuple[Optional[float], Optional[int]]] = {}
rt_cache: Dict[str, Tuple[Optional[int], Optional[int], Optional[str]]] = {}

movie_rows = []
for title in sorted(df_show["movie_title"].unique()):
    rt_aud, rt_crit, rt_url = rt_get_scores(session, title, rt_cache, debug=DEBUG_RT)
    imdb, runtime_min = imdb_rating_and_runtime(session, title, imdb_cache)

    rank, prim_score, prim_src = pick_primary(rt_aud, imdb, rt_crit)

    movie_rows.append({
        "movie_title": title,
        "runtime": fmt_runtime(runtime_min),
        "runtime_min": runtime_min,  # hidden helper
        "rt_audience": rt_aud,
        "imdb_rating": imdb,
        "rt_critic": rt_crit,
        "sort_rank": rank,
        "primary_score": prim_score,
        "primary_source": prim_src,
        "rt_url": rt_url,
    })
    time.sleep(0.15)

df_movies = pd.DataFrame(movie_rows)

# Aggregate showtimes into one cell per movie + a_list_excluded_any (ANY showtime excluded)
agg = []
for title, df_st in df_show.groupby("movie_title"):
    agg.append({
        "movie_title": title,
        "a_list_excluded_any": bool(df_st["a_list_excluded"].fillna(False).any()),
        "showtimes": build_showtimes_cell(df_st),
    })
df_agg = pd.DataFrame(agg)

df_summary = (
    df_movies.merge(df_agg, on="movie_title", how="left")
            .sort_values(
                by=["sort_rank", "primary_score", "movie_title"],
                ascending=[True, False, True]
            )
            .reset_index(drop=True)
)

# Make title clickable to RT when we found a URL
def link_title(row):
    u = row.get("rt_url")
    if isinstance(u, str) and u.startswith("http"):
        return f'<a href="{u}" target="_blank" rel="noopener noreferrer">{row["movie_title"]}</a>'
    return row["movie_title"]

df_display = df_summary.copy()
df_display["movie_title"] = df_display.apply(link_title, axis=1)

# Hide helper columns you probably don't want to see
df_display = df_display.drop(columns=["sort_rank", "rt_url", "runtime_min"])

# Optional: nicer column order
desired = [
    "movie_title",
    "runtime",
#    "a_list_excluded_any",
    "rt_audience",
    "imdb_rating",
    "rt_critic",
#    "primary_score",
#    "primary_source",
    "showtimes",
]
df_display = df_display[[c for c in desired if c in df_display.columns]]

pd.set_option("display.max_colwidth", None)

print(f"Upcoming weekend (Pacific): {sat.isoformat()} (Sat), {sun.isoformat()} (Sun)")
html_out = df_display.to_html(index=False, escape=False).replace("\\n", "<br>")

css = """
<style>
table.dataframe th, table.dataframe td {
  text-align: left !important;
  vertical-align: top;
}
</style>
"""

display(HTML(css + html_out))


Upcoming weekend (Pacific): 2025-12-20 (Sat), 2025-12-21 (Sun)


movie_title,runtime,rt_audience,imdb_rating,rt_critic,showtimes
Kill Bill: The Whole Bloody Affair,4h 35m,99.0,8.8,100.0,AMC Orange 30• 2025-12-20: 8:30 am⛔ [Laser at AMC]
Christina Aguilera: Christmas in Paris,1h 25m,97.0,9.5,NaN,AMC Bay Street 16• 2025-12-21: 7:00 pm⛔ [Laser at AMC]AMC Orange 30• 2025-12-21: 7:00 pm⛔ [Laser at AMC]AMC Tustin 14 @ The District• 2025-12-21: 7:00 pm⛔ [Laser at AMC]
Dhurandhar,3h 34m,96.0,8.6,54.0,"AMC Orange 30• 2025-12-20: 9:05 am [Hindi Spoken with English Subtitles], 10:00 am [Hindi Spoken with English Subtitles], 12:20 pm [Hindi Spoken with English Subtitles], 2:10 pm [Hindi Spoken with English Subtitles], 3:25 pm [Hindi Spoken with English Subtitles], 6:45 pm [Hindi Spoken with English Subtitles], 9:30 pm [Hindi Spoken with English Subtitles]• 2025-12-21: 9:05 am [Hindi Spoken with English Subtitles], 12:20 pm [Hindi Spoken with English Subtitles], 3:25 pm [Hindi Spoken with English Subtitles], 9:30 pm [Hindi Spoken with English Subtitles]"
Zootopia 2,1h 48m,96.0,7.7,91.0,"AMC Bay Street 16• 2025-12-20: 10:40 am [Laser at AMC], 1:30 pm [Laser at AMC], 4:20 pm [Laser at AMC], 7:10 pm [Laser at AMC], 10:00 pm [Laser at AMC]• 2025-12-21: 10:40 am [Laser at AMC], 1:30 pm [Laser at AMC], 4:20 pm [Laser at AMC], 7:10 pm [Laser at AMC], 10:00 pm [Laser at AMC]AMC Orange 30• 2025-12-20: 8:30 am [Laser at AMC], 11:15 am [Laser at AMC], 2:10 pm [Laser at AMC], 4:35 pm [Laser at AMC], 7:50 pm [Laser at AMC], 9:20 pm [RealD 3D], 10:20 pm [Laser at AMC]• 2025-12-21: 8:30 am [Laser at AMC], 11:15 am [Laser at AMC], 2:10 pm [Laser at AMC], 4:35 pm [Laser at AMC], 7:50 pm [Laser at AMC], 9:10 pm [RealD 3D], 10:10 pm [Laser at AMC]AMC Tustin 14 @ The District• 2025-12-20: 9:50 am [Laser at AMC], 12:55 pm [Laser at AMC], 4:10 pm [Laser at AMC], 7:00 pm [Laser at AMC], 9:35 pm [Laser at AMC]• 2025-12-21: 9:30 am [Laser at AMC], 12:15 pm [Laser at AMC], 3:10 pm [Laser at AMC], 7:35 pm [Laser at AMC], 9:35 pm [Laser at AMC]AMC Woodbridge 5• 2025-12-20: 10:30 am [Laser at AMC], 12:15 pm [Laser at AMC], 3:45 pm [Laser at AMC], 7:30 pm [Laser at AMC], 10:15 pm [Laser at AMC]• 2025-12-21: 10:30 am [Laser at AMC], 12:15 pm [Laser at AMC], 3:45 pm [Laser at AMC], 7:30 pm [Laser at AMC], 10:15 pm [Laser at AMC]"
Merrily We Roll Along,2h 25m,95.0,8.3,94.0,AMC Orange 30• 2025-12-20: 8:40 am⛔ [Laser at AMC]• 2025-12-21: 8:45 am⛔ [Laser at AMC]
Predator: Badlands,1h 47m,95.0,7.4,86.0,"AMC Orange 30• 2025-12-20: 8:40 am [Laser at AMC], 7:40 pm [Laser at AMC]• 2025-12-21: 8:40 am [Laser at AMC], 7:40 pm [Laser at AMC]"
Sentimental Value,2h 13m,94.0,7.9,96.0,AMC Orange 30• 2025-12-20: 11:20 pm [Norwegian Spoken with English Subtitles]• 2025-12-21: 11:20 pm [Norwegian Spoken with English Subtitles]
Wicked: For Good,2h 17m,93.0,7.0,67.0,"AMC Bay Street 16• 2025-12-20: 11:00 am [Laser at AMC], 2:20 pm [Laser at AMC], 5:40 pm [Laser at AMC], 9:00 pm [Laser at AMC]• 2025-12-21: 11:00 am [Laser at AMC], 2:20 pm [Laser at AMC], 5:40 pm [Laser at AMC], 9:00 pm [Laser at AMC]AMC Orange 30• 2025-12-20: 8:50 am [Laser at AMC], 12:05 pm [Laser at AMC], 3:20 pm [Laser at AMC], 6:50 pm [Laser at AMC], 10:15 pm [Laser at AMC]• 2025-12-21: 8:50 am [Laser at AMC], 12:05 pm [Laser at AMC], 3:25 pm [Laser at AMC], 6:50 pm [Laser at AMC], 10:15 pm [Laser at AMC]AMC Tustin 14 @ The District• 2025-12-20: 12:55 pm [Laser at AMC], 4:20 pm [Laser at AMC], 7:05 pm [Laser at AMC], 10:35 pm [Laser at AMC]• 2025-12-21: 12:40 pm [Laser at AMC], 4:00 pm [Laser at AMC], 7:00 pm [Laser at AMC], 10:25 pm [Laser at AMC]AMC Woodbridge 5• 2025-12-20: 1:15 pm [Laser at AMC], 4:10 pm [Laser at AMC]• 2025-12-21: 1:15 pm [Laser at AMC], 4:10 pm [Laser at AMC]"
Hamnet,2h 5m,92.0,8.1,86.0,"AMC Bay Street 16• 2025-12-20: 9:50 am [Laser at AMC], 4:00 pm [Laser at AMC]• 2025-12-21: 1:00 pm [Laser at AMC]AMC Orange 30• 2025-12-20: 4:20 pm [Laser at AMC], 7:25 pm [Laser at AMC]• 2025-12-21: 4:20 pm [Laser at AMC], 7:25 pm [Laser at AMC]AMC Tustin 14 @ The District• 2025-